
1. What is Unity Catalog?
Interview definition:

Unity Catalog is the centralized governance and security layer in Databricks used to manage data assets, permissions, access control, auditing and lineage across a Databricks environment.

It helps organizations control who can access data, what they can access and what operations they can perform.
Main responsibilities
1. Centralized data governance.
2. Access control and permissions.
3. Table and column-level security.
4. Row-level security.
5. Data lineage.
6. External storage management.
7. Data discovery and auditing.
8. Cross-workspace data governance.








medium.com


Real-time example:

In your E-Commerce Lakehouse project:
- Data engineers need access to Bronze and Silver.
- Business analysts need access to Gold.
- Finance employees should access payment information.
- Customer personal information should be restricted.
- Managers need access to business dashboards.
Unity Catalog allows you to implement these access rules centrally.
2. Unity Catalog hierarchy
The hierarchy is:
Metastore

Top-level governance container


Catalog

Collection of schemas


Schema

Collection of tables, views and other objects


Table / View / Volume

Data objects and storage access


2.1 Metastore
Definition: A metastore is the top-level Unity Catalog governance container that manages metadata and permissions for its registered data assets.

Example:
SHOW CATALOGS;


2.2 Catalog
Definition: A catalog is a logical container that organizes schemas.

Example:
CREATE CATALOG ecommerce;


2.3 Schema
Definition: A schema is a logical container inside a catalog that organizes tables, views and other database objects.

Example:
CREATE SCHEMA ecommerce.bronze;


2.4 Table
Definition: A table stores structured data and is registered inside a schema.

Example:
CREATE TABLE ecommerce.bronze.customers (
    CustomerID INT,
    CustomerName STRING,
    Email STRING
);


2.5 Three-level namespace
The standard table naming format is:
catalog.schema.table


Your project example:
workspace.ecommerce_bronze.customers

workspace.ecommerce_silver.customers

workspace.ecommerce_gold.daily_sales


Interview question: What is the difference between a catalog and a schema?

Answer: A catalog is a higher-level container that organizes schemas, whereas a schema organizes tables, views, volumes and other objects.
3. Unity Catalog permissions
Interview definition:

Permissions determine which users or groups can access, read, modify or manage Databricks data objects.

Unity Catalog uses SQL commands such as:
- GRANT
- REVOKE
- SHOW GRANTS
3.1 Important privileges
Privilege	Purpose
USE CATALOG	Access a catalog
USE SCHEMA	Access a schema
SELECT	Read table data
MODIFY	Insert, update and delete table data
CREATE TABLE	Create tables
CREATE SCHEMA	Create schemas
CREATE VOLUME	Create volumes
READ VOLUME	Read volume files
WRITE VOLUME	Write volume files
MANAGE	Manage an object's permissions and ownership

3.2 How to grant access
Scenario: Give the Data Analyst permission to read Gold data.
GRANT USE CATALOG
ON CATALOG workspace
TO `data_analysts`;


GRANT USE SCHEMA
ON SCHEMA workspace.ecommerce_gold
TO `data_analysts`;


GRANT SELECT
ON SCHEMA workspace.ecommerce_gold
TO `data_analysts`;


Now the analysts can read tables in the Gold schema, subject to any additional security policies.
3.3 How to revoke access
REVOKE SELECT
ON SCHEMA workspace.ecommerce_gold
FROM `data_analysts`;


3.4 How to view permissions
SHOW GRANTS ON CATALOG workspace;


SHOW GRANTS ON SCHEMA workspace.ecommerce_gold;


SHOW GRANTS ON TABLE
workspace.ecommerce_gold.daily_sales;


Important interview point: To read a table, a user generally needs all three:
1. USE CATALOG
2. USE SCHEMA
3. SELECT on the table (or inherited SELECT permission).
Unity Catalog supports privilege inheritance, so permissions granted at higher levels can apply to child objects. 
Databricks on AWS+2


4. Row-level security
Interview definition:

Row-level security restricts which rows a user can see within a table based on their identity, group membership or business rules.

Real-time example:

Suppose the E-Commerce company operates in three regions.
OrderID	Region	Amount
101	Chennai	500
102	Bangalore	700
103	Mumbai	900

Requirements:
- Chennai team sees Chennai records.
- Bangalore team sees Bangalore records.
- Admin sees all records.
We can implement a row filter using a SQL function.
Step 1: Create a security function
CREATE FUNCTION workspace.ecommerce_gold.region_filter(region STRING)
RETURNS BOOLEAN
RETURN
    CASE
        WHEN IS_ACCOUNT_GROUP_MEMBER('ecommerce_admins')
            THEN TRUE

        WHEN IS_ACCOUNT_GROUP_MEMBER('chennai_team')
             AND region = 'Chennai'
            THEN TRUE

        WHEN IS_ACCOUNT_GROUP_MEMBER('bangalore_team')
             AND region = 'Bangalore'
            THEN TRUE

        ELSE FALSE
    END;


Step 2: Apply the row filter
ALTER TABLE workspace.ecommerce_gold.daily_sales
SET ROW FILTER
workspace.ecommerce_gold.region_filter
ON (Region);


Step 3: Query the table
SELECT *
FROM workspace.ecommerce_gold.daily_sales;


The result is automatically filtered according to the querying user's group membership.

Interview point: Users query the same table, but they receive different rows based on their access rules.

The function must be created with the appropriate privileges, and the relevant account groups must exist. 
Microsoft Learn+1

5. Column-level security
Interview definition:

Column-level security controls which column values a user can see. Unity Catalog supports column masks to return either original or masked values.

Real-time example:

An e-commerce customer table contains:
CustomerID	Name	Email
101	John	john@gmail.com
102	David	david@gmail.com

Requirements:
- Admin can see complete email addresses.
- Other users should see masked email addresses.
Step 1: Create a masking function
CREATE FUNCTION workspace.ecommerce_gold.mask_email(email STRING)
RETURNS STRING
RETURN
    CASE
        WHEN IS_ACCOUNT_GROUP_MEMBER('ecommerce_admins')
            THEN email
        ELSE CONCAT(
            'XXXXXX',
            SUBSTRING(email, INSTR(email, '@'))
        )
    END;


Step 2: Apply column masking
ALTER TABLE workspace.ecommerce_gold.customers
ALTER COLUMN Email
SET MASK workspace.ecommerce_gold.mask_email;


Step 3: Query
SELECT CustomerID, Name, Email
FROM workspace.ecommerce_gold.customers;


Non-admin users see masked email values, while authorized administrators see the original values.
Row security vs Column security
Row-level security	Column-level security
Restricts rows	Masks column values
Controls which records are visible	Controls which values are visible
Example: Region-based access	Example: Email masking
Uses row filters	Uses column masks

For larger organizations, Databricks also supports Attribute-Based Access Control (ABAC), which can centralize policies across tagged tables and schemas. 
Databricks on AWS+1


6. What is Data Lineage?
Interview definition:

Data lineage tracks where data originates, how it moves through different transformations and where it is consumed.

It helps data engineers understand dependencies, troubleshoot issues and audit data transformations.

Real-time example – E-Commerce Lakehouse:
Source

MySQL Orders


Bronze

ecommerce_bronze.orders


Silver

ecommerce_silver.orders


Gold

ecommerce_gold.daily_sales


Dashboard

Daily Revenue KPI


How to view lineage
1. Open Databricks workspace.
2. Navigate to Catalog.
3. Open the required catalog.
4. Select the table.
5. Open the Lineage tab.
6. Inspect upstream and downstream dependencies.
Upstream lineage: Where did the data come from?

Downstream lineage: Which tables, notebooks or dashboards use this data?

Example interview question:

Q: Suppose a Gold dashboard displays incorrect revenue. How does lineage help?

Answer:

I trace the lineage backward from the Gold KPI table to Silver and Bronze. I identify the transformation or source responsible for the incorrect calculation and investigate the affected pipeline.

Unity Catalog captures supported data lineage across registered data assets and provides lineage exploration through Catalog Explorer. 
Databricks on Google Cloud


7. What are External Locations?
Interview definition:

An External Location is a Unity Catalog securable object that combines a cloud storage path with a storage credential to control access to external cloud storage.

Examples:
- AWS S3
- Azure Data Lake Storage
- Google Cloud Storage
Real-time example:

An e-commerce company stores CSV files in Azure Data Lake Storage. Databricks needs controlled access to these files.

Instead of hardcoding cloud credentials in notebooks, administrators configure a storage credential and register the storage path as an external location.
7.1 External Location architecture








linkedin.com


7.2 Create an External Location
Example SQL (requires an already configured storage credential):
CREATE EXTERNAL LOCATION ecommerce_landing_location

URL 'abfss://landing@storageaccount.dfs.core.windows.net/ecommerce'

WITH (STORAGE CREDENTIAL ecommerce_storage_credential);


7.3 Grant access
GRANT READ FILES
ON EXTERNAL LOCATION ecommerce_landing_location
TO `data_engineers`;


To grant write access, grant both READ FILES and WRITE FILES.

For ordinary file operations, Databricks recommends using Unity Catalog volumes and their READ VOLUME and WRITE VOLUME privileges rather than granting direct file access. 
Databricks on AWS+2

8. What is a Volume in Unity Catalog?
Interview definition:

A Volume is a Unity Catalog object used to govern access to non-tabular files such as CSV, JSON, Parquet, images and text files.
8.1 Volume path structure
/Volumes/catalog/schema/volume/


Your project example:
/Volumes/workspace/default/data_volume/ecommerce_lakehouse/


8.2 Create a Volume
CREATE VOLUME workspace.default.data_volume;


8.3 List files
display(    dbutils.fs.ls(        "/Volumes/workspace/default/data_volume/"    ))



8.4 Read a CSV file
df = spark.read \    .option("header", "true") \    .csv(        "/Volumes/workspace/default/data_volume/"        "ecommerce_lakehouse/landing/customers/customers.csv"    )display(df)



8.5 Volume vs Table
Volume	Table
Stores non-tabular files	Stores structured data
Accessed using file paths	Accessed using table names
CSV, JSON, images, documents	Rows and columns
/Volumes/...	catalog.schema.table

Interview point: Your E-Commerce Lakehouse uses a Unity Catalog Volume to store landing files, schema information, checkpoints and logs. Delta tables store the Bronze, Silver and Gold data.

Official path conventions: 
Microsoft Learn+1


9. What is a Connection in Unity Catalog?
Interview definition:

A Connection is a Unity Catalog securable object that stores connection information and authentication configuration required to communicate with an external system.

Examples:
- MySQL
- PostgreSQL
- Snowflake
- Other supported external database systems
Real-time example:

Your e-commerce company maintains customer and order information in MySQL.

Instead of copying the database immediately into Databricks, you configure a connection that allows Databricks to communicate with MySQL.

A connection stores information such as:
- Hostname.
- Port number.
- Database connection details.
- Authentication configuration.
It does not store a copy of the database's table data.

Security recommendation: Never hardcode actual passwords in notebooks or SQL statements. Use Databricks secret scopes and the secret() function for credential references. 
Databricks on AWS+1

10. What is Lakehouse Federation?
Interview definition:

Lakehouse Federation allows Databricks to query data from external database systems without first copying the entire dataset into Databricks.

It uses Unity Catalog connections and foreign catalogs.








linkedin.com


How it works
1. Create a connection to an external database.
2. Register a foreign catalog.
3. Grant permissions to users.
4. Query external tables using SQL.
5. Databricks executes federated queries against the external database.
This is useful when you want to access external data without duplicating it. 
Databricks on AWS+1

11. What is a Foreign Catalog?
Interview definition:

A Foreign Catalog is a Unity Catalog object that represents the schemas and tables of an external database.

It provides a Databricks interface for querying remote database tables.

Important characteristics:
- Metadata is registered in Unity Catalog.
- Actual table data remains in the external database.
- Queries access the remote source through federation.
- No initial full data copy is required.
- Unity Catalog permissions can govern access to the foreign catalog.
11.1 Example
Suppose your company has a MySQL database:
MySQL Database
      |
      |-- customers
      |-- orders
      |-- payments
      |-- products


You register it as a Foreign Catalog:
Databricks Unity Catalog

mysql_ecommerce
      |
      |-- public
             |
             |-- customers
             |-- orders
             |-- payments
             |-- products


You can now query the external tables using three-level naming.
SELECT *
FROM mysql_ecommerce.public.customers;


The data remains in MySQL. Databricks queries it through Lakehouse Federation.

Important distinction: This is federated querying, not guaranteed real-time streaming. Results are obtained by querying the external source, subject to source availability, network latency, query pushdown and source database performance. 
Databricks on AWS+1


12. Practical: Connect MySQL to Databricks
Important Practical Interview Question
Interviewer question:

How do you connect an external MySQL database to Databricks using Unity Catalog and query the tables without copying the data?

Let's implement it step by step.
Step 1: Prerequisites
You need:
- Unity Catalog-enabled Databricks workspace.
- MySQL database.
- MySQL hostname and port.
- Database username and password.
- Network connectivity from Databricks compute to MySQL.
- Required Unity Catalog permissions.
For MySQL, Lakehouse Federation requires supported compute, including SQL warehouses or compatible Databricks Runtime compute. 
Databricks on AWS

Step 2: Create MySQL connection
Example SQL:
CREATE CONNECTION mysql_ecommerce_connection

TYPE MYSQL

OPTIONS (
    host 'mysql-server.example.com',
    port '3306',
    user 'databricks_user',
    password secret('ecommerce_scope', 'mysql_password')
);


Replace the hostname and secret references with your actual configured values.

The secret scope and secret must already exist and be accessible under the relevant security configuration.
Step 3: Create Foreign Catalog
CREATE FOREIGN CATALOG mysql_ecommerce

USING CONNECTION mysql_ecommerce_connection

OPTIONS (
    database 'ecommerce_db'
);


This registers the external MySQL database in Unity Catalog.
Step 4: View catalogs
SHOW CATALOGS;


Step 5: Inspect schemas
SHOW SCHEMAS IN mysql_ecommerce;


Step 6: Inspect tables
Assuming the remote schema is public:
SHOW TABLES IN mysql_ecommerce.public;


Step 7: Query MySQL data
SELECT *
FROM mysql_ecommerce.public.customers;


Step 8: Perform aggregation
SELECT
    COUNT(*) AS total_customers
FROM mysql_ecommerce.public.customers;


Expected result
You can query MySQL data through Databricks SQL without first copying the complete MySQL database into Delta tables.

The actual schema name depends on your MySQL database and connector metadata.

Official setup reference: Databricks – Run federated queries on MySQL.

13. Practical: Connect PostgreSQL to Databricks
The workflow is almost identical, but the connection type and port are different.
Step 1: Create PostgreSQL connection
CREATE CONNECTION postgres_ecommerce_connection

TYPE POSTGRESQL

OPTIONS (
    host 'postgres-server.example.com',
    port '5432',
    user 'databricks_user',
    password secret('ecommerce_scope', 'postgres_password')
);


Step 2: Create Foreign Catalog
CREATE FOREIGN CATALOG postgres_ecommerce

USING CONNECTION postgres_ecommerce_connection

OPTIONS (
    database 'ecommerce_db'
);


Step 3: View tables
SHOW TABLES IN postgres_ecommerce.public;


Step 4: Query data
SELECT *
FROM postgres_ecommerce.public.orders;


Step 5: Join local and foreign data
Suppose you have a local Delta table and a remote PostgreSQL table.
SELECT
    p.OrderID,
    p.TotalAmount,
    s.CustomerID
FROM postgres_ecommerce.public.orders p

JOIN workspace.ecommerce_silver.customers s

ON p.CustomerID = s.CustomerID;


This demonstrates a federated query across an external database and a local Delta table.

Official reference: Databricks – Run federated queries on PostgreSQL.
14. Foreign Catalog vs External Location
Frequently Asked Comparison
This is one of the most important distinctions in your syllabus.
Feature	Foreign Catalog	External Location
Purpose	Access external databases	Govern cloud storage paths
Source	MySQL, PostgreSQL	S3, ADLS, GCS
Main object	Remote database metadata	Storage path + credential
Data copy	Not required	Not required merely to register location
Query access	Federated SQL	File and table operations
Authentication	Database credentials	Cloud storage credentials
Example	MySQL orders	CSV files in ADLS

Simple interview answer:

A Foreign Catalog is used to access external database tables without copying the data into Databricks. An External Location is used to govern access to external cloud storage paths using storage credentials.

Do not confuse these two objects.
15. When should you use a Foreign Catalog?
There are three important scenarios.

Scenario 1 – Ad-hoc analysis

You want to investigate MySQL customer data without building a complete ingestion pipeline.

Use a Foreign Catalog.

Scenario 2 – Incremental data extraction

You want to extract only recent orders from PostgreSQL into your Bronze Delta table.

You can query the source through federation and write the selected records into Delta.

Example:
CREATE TABLE workspace.ecommerce_bronze.recent_orders
AS
SELECT *
FROM mysql_ecommerce.public.orders
WHERE OrderDate >= DATE '2026-11-01';


This example creates a local Delta table from the selected source records. It is a snapshot of that query, not an automatically maintained incremental pipeline.

For recurring incremental ingestion, use a watermark or CDC strategy and orchestrate subsequent loads.

Scenario 3 – Avoid unnecessary duplication

You want to analyze external data without maintaining another complete copy.

Use a Foreign Catalog and query the remote tables directly.
Foreign Catalog vs copying data into Delta
Foreign Catalog	Delta ingestion
No initial data copy	Data copied into Databricks
Queries remote database	Queries local Delta tables
Depends on source availability	Local copy can remain available
Source performance matters	Can optimize local storage
Useful for direct access	Useful for large-scale transformations

My recommendation: Use federation for ad-hoc access and selective queries. Use Delta ingestion for repeatable production transformations, historical storage, large-scale analytics and performance-sensitive workloads.

16. Important Unity Catalog SQL commands – Interview Revision
Keep this table for quick revision.
Requirement	SQL command
Show catalogs	SHOW CATALOGS;
Create catalog	CREATE CATALOG ecommerce;
Show schemas	SHOW SCHEMAS IN ecommerce;
Create schema	CREATE SCHEMA ecommerce.bronze;
Show tables	SHOW TABLES IN ecommerce.bronze;
Describe table	DESCRIBE TABLE ecommerce.bronze.orders;
Grant access	GRANT SELECT ...
Revoke access	REVOKE SELECT ...
View grants	SHOW GRANTS ON TABLE ...;
Create volume	CREATE VOLUME ...;
Show volumes	SHOW VOLUMES IN ...;
Show connections	SHOW CONNECTIONS;
Describe connection	DESCRIBE CONNECTION ...;
Create connection	CREATE CONNECTION ...;
Create foreign catalog	CREATE FOREIGN CATALOG ...;
Show lineage	Catalog Explorer → Table → Lineage

17. Top 15 Unity Catalog interview questions
No.	Interview question	Key answer
1	What is Unity Catalog?	Centralized governance
2	What is a Metastore?	Top-level governance container
3	What is a Catalog?	Collection of schemas
4	What is a Schema?	Collection of database objects
5	What is the three-level namespace?	Catalog.Schema.Table
6	What is GRANT?	Provides permissions
7	What is REVOKE?	Removes permissions
8	What is row-level security?	Restricts records
9	What is column-level security?	Masks sensitive values
10	What is Data Lineage?	Tracks data dependencies
11	What is an External Location?	Governed cloud storage path
12	What is a Volume?	Governed non-tabular file storage
13	What is a Connection?	External system connection configuration
14	What is a Foreign Catalog?	Metadata representation of remote database objects
15	What is Lakehouse Federation?	Query external data without initially copying it


Unity Catalog – 8 Practical Interview Scenarios with Answers
Target: Databricks Data Engineer – 1.5–2 Years Experience

These are scenario-based questions. In an interview, don't just define the concept. Explain the business requirement, the Databricks feature, the implementation and why you chose it.

Q1. A finance analyst should see Gold data but not customer email addresses. How would you implement this?
Important – Column Security
Business requirement:

The finance team needs access to customer and sales information, but customer email addresses are confidential.

Solution: Unity Catalog Column Masking.

Step 1: Grant access to the Finance group.
GRANT USE CATALOG
ON CATALOG workspace
TO `finance_team`;

GRANT USE SCHEMA
ON SCHEMA workspace.ecommerce_gold
TO `finance_team`;

GRANT SELECT
ON SCHEMA workspace.ecommerce_gold
TO `finance_team`;


Step 2: Create a masking function.
CREATE FUNCTION workspace.ecommerce_gold.mask_email(email STRING)
RETURNS STRING
RETURN
    CASE
        WHEN IS_ACCOUNT_GROUP_MEMBER('finance_admins')
        THEN email

        ELSE CONCAT(
            'XXXXXX',
            SUBSTRING(email, INSTR(email, '@'))
        )
    END;


Step 3: Apply the mask.
ALTER TABLE workspace.ecommerce_gold.customers
ALTER COLUMN Email
SET MASK workspace.ecommerce_gold.mask_email;


Step 4: Test.
SELECT
    CustomerID,
    CustomerName,
    Email
FROM workspace.ecommerce_gold.customers;


Expected output:
CustomerID	CustomerName	Email
101	John	XXXXXX@gmail.com
102	David	XXXXXX@yahoo.com

Interview answer:

I would use Unity Catalog column masking. I would create a masking function and apply it to the Email column. Finance analysts could access the Gold table but would see masked email addresses, while authorized administrators could see the original values.

Q2. A regional manager should see only Chennai orders. How would you implement row-level security?
Important – Row Security
Business requirement:

The company operates in Chennai, Bangalore and Mumbai.

Each regional manager should access only their region's orders.

Solution: Unity Catalog Row Filters.

Step 1: Create a row-filter function.
CREATE FUNCTION workspace.ecommerce_gold.region_filter(
    region STRING
)
RETURNS BOOLEAN

RETURN
    CASE
        WHEN IS_ACCOUNT_GROUP_MEMBER('ecommerce_admins')
            THEN TRUE

        WHEN IS_ACCOUNT_GROUP_MEMBER('chennai_team')
             AND region = 'Chennai'
            THEN TRUE

        ELSE FALSE
    END;


Step 2: Apply the row filter.
ALTER TABLE workspace.ecommerce_gold.daily_sales

SET ROW FILTER
workspace.ecommerce_gold.region_filter
ON (Region);


Step 3: Query the table.
SELECT *
FROM workspace.ecommerce_gold.daily_sales;


Expected result for a Chennai manager:
OrderID	Region	Amount
101	Chennai	500
105	Chennai	800

Bangalore and Mumbai records will not be returned.

Interview answer:

I would implement row-level security using Unity Catalog row filters. I would create a SQL function based on group membership and region, then attach that function to the table. This ensures each regional manager sees only authorized records.

Q3. Your organization has MySQL data and wants to query it without copying it. What would you implement?
Important – Lakehouse Federation
Business requirement:

The company stores orders in MySQL. Data analysts want to query those orders from Databricks without copying the entire database.

Solution: Lakehouse Federation using a Connection and Foreign Catalog.

Step 1: Create a MySQL connection.
CREATE CONNECTION mysql_connection

TYPE MYSQL

OPTIONS (
    host 'mysql-server.example.com',
    port '3306',
    user 'databricks_user',
    password secret('ecommerce_scope', 'mysql_password')
);


Step 2: Create a Foreign Catalog.
CREATE FOREIGN CATALOG mysql_ecommerce

USING CONNECTION mysql_connection

OPTIONS (
    database 'ecommerce_db'
);


Step 3: View schemas.
SHOW SCHEMAS IN mysql_ecommerce;


Step 4: View tables.
SHOW TABLES IN mysql_ecommerce.public;


Step 5: Query external data.
SELECT *
FROM mysql_ecommerce.public.orders;


Interview answer:

I would use Databricks Lakehouse Federation. First, I would create a MySQL connection and then register a Foreign Catalog. This allows users to query external MySQL tables through Databricks SQL without initially copying the data into Delta tables.

Q4. What is the difference between a Foreign Catalog and an External Location?
Business requirement:

Your company has:
- MySQL customer information.
- CSV files stored in Azure Data Lake Storage.
Which Unity Catalog objects would you use?

Answer:
Foreign Catalog	External Location
Connects to external databases	Controls cloud storage access
MySQL, PostgreSQL	S3, ADLS, GCS
Uses a Connection	Uses a Storage Credential
Exposes remote schemas and tables	Registers a cloud storage path
Supports federated queries	Supports governed file access

Example:

Foreign Catalog:
SELECT *
FROM mysql_ecommerce.public.customers;


External Location:
CREATE EXTERNAL LOCATION ecommerce_location

URL 'abfss://landing@storageaccount.dfs.core.windows.net/ecommerce'

WITH (
    STORAGE CREDENTIAL ecommerce_storage_credential
);


Interview answer:

I use a Foreign Catalog when I need to access tables in external databases without initially copying their data. I use an External Location when I need to govern access to cloud storage paths using Unity Catalog storage credentials.

Remember:
- Foreign Catalog = External database.
- External Location = Cloud storage path.

Q5. A user can see a catalog but cannot query a table. How would you troubleshoot the permissions?
Important – Access Troubleshooting
Business requirement:

A data analyst can see the workspace catalog but receives a permission error when querying:
SELECT *
FROM workspace.ecommerce_gold.daily_sales;


Possible reasons:
1. Missing USE CATALOG.
2. Missing USE SCHEMA.
3. Missing SELECT.
4. Access is denied through another applicable policy.
5. The user is not a member of the required group.
Step 1: Check catalog permissions.
SHOW GRANTS ON CATALOG workspace;


Step 2: Check schema permissions.
SHOW GRANTS ON SCHEMA workspace.ecommerce_gold;


Step 3: Check table permissions.
SHOW GRANTS ON TABLE
workspace.ecommerce_gold.daily_sales;


Step 4: Grant the required privileges.
GRANT USE CATALOG
ON CATALOG workspace
TO `data_analysts`;


GRANT USE SCHEMA
ON SCHEMA workspace.ecommerce_gold
TO `data_analysts`;


GRANT SELECT
ON TABLE workspace.ecommerce_gold.daily_sales
TO `data_analysts`;


Step 5: Test again.
SELECT *
FROM workspace.ecommerce_gold.daily_sales;


Interview answer:

First, I would check the user's effective permissions at catalog, schema and table levels using SHOW GRANTS. Then I would identify the missing privilege and grant the minimum required access. I would also check group membership and applicable row filters or column masks.

Q6. How do you trace an incorrect Gold KPI back to its source?
Important – Data Lineage
Business requirement:

The executive dashboard displays incorrect daily revenue.

You need to identify whether the issue originated in Bronze, Silver or Gold.

Solution: Unity Catalog Data Lineage.

Example architecture:
MySQL Orders

Source data


Bronze Orders

Raw ingestion


Silver Orders

Cleaning and validation


Gold Daily Sales

Revenue aggregation


Executive Dashboard

Revenue KPI


Investigation steps:
1. Open Catalog Explorer.
2. Select workspace.ecommerce_gold.daily_sales.
3. Open the Lineage tab.
4. Identify upstream tables and transformations.
5. Trace the data back to Silver and Bronze.
6. Check whether duplicates, incorrect amounts or transformation errors caused the issue.
7. Fix the transformation and rerun the affected pipeline.
Example SQL:
SELECT
    OrderDate,
    COUNT(*) AS TotalOrders,
    SUM(TotalAmount) AS TotalRevenue
FROM workspace.ecommerce_silver.orders
GROUP BY OrderDate;


Compare the Silver aggregation with the Gold KPI.

Interview answer:

I would use Unity Catalog data lineage to identify upstream dependencies of the Gold table. Then I would validate the Silver and Bronze records, inspect the transformation logic and identify where the incorrect calculation originated.

Q7. How do you securely connect PostgreSQL to Databricks without hardcoding credentials?
Business requirement:

Your company has PostgreSQL running on an external server. You need to query its data securely from Databricks.

Solution: Unity Catalog Connection + Secret Management + Foreign Catalog.

Step 1: Store credentials securely.

Example secret retrieval:
username = dbutils.secrets.get(    scope="ecommerce_scope",    key="postgres_username")password = dbutils.secrets.get(    scope="ecommerce_scope",    key="postgres_password")



Do not print these values or place them in notebooks or GitHub.

Step 2: Create the connection.
CREATE CONNECTION postgres_connection

TYPE POSTGRESQL

OPTIONS (
    host 'postgres-server.example.com',
    port '5432',
    user 'databricks_user',
    password secret('ecommerce_scope', 'postgres_password')
);


Step 3: Create a Foreign Catalog.
CREATE FOREIGN CATALOG postgres_ecommerce

USING CONNECTION postgres_connection

OPTIONS (
    database 'ecommerce_db'
);


Step 4: Query PostgreSQL.
SELECT *
FROM postgres_ecommerce.public.orders;


Interview answer:

I would store credentials in Databricks secret management and reference the password through the supported SQL secret function when creating the Unity Catalog connection. Then I would register a Foreign Catalog and query the PostgreSQL tables through Lakehouse Federation.

Q8. When would you use a Foreign Catalog instead of loading data into Bronze?
Business requirement:

Your company has 10 million customer records in MySQL.

The business wants to analyze the data from Databricks.

You have two options.

Option A: Foreign Catalog
SELECT *
FROM mysql_ecommerce.public.customers;


Advantages:
- No initial full data copy.
- Quick access to external tables.
- Useful for ad-hoc analysis.
- Avoids unnecessary duplication.
Disadvantages:
- Depends on external database availability.
- Query performance depends on the source.
- Large analytical queries can put pressure on the source database.
Option B: Load into Bronze
df = spark.read.format("jdbc") \    .option("url", jdbc_url) \    .option("dbtable", "customers") \    .option("user", username) \    .option("password", password) \    .option("driver", "com.mysql.cj.jdbc.Driver") \    .load()df.write.format("delta") \    .mode("append") \    .saveAsTable("workspace.ecommerce_bronze.customers")



This JDBC example illustrates a batch ingestion approach, not a production-ready incremental CDC implementation.

Advantages:
- Local Delta storage.
- Historical data retention.
- Better control over transformation and optimization.
- Suitable for downstream Medallion processing.
Disadvantages:
- Additional storage.
- Data synchronization required.
- Ingestion pipeline maintenance.
Comparison
Requirement	Recommended choice
Ad-hoc querying	Foreign Catalog
Avoid data duplication	Foreign Catalog
Historical storage	Bronze Delta
Large-scale transformations	Delta tables
Repeated dashboard queries	Gold Delta
Incremental CDC pipeline	Bronze Delta + CDC/MERGE

Interview answer:

I would choose a Foreign Catalog for ad-hoc analysis and direct access to external data without copying it. If the business requires historical storage, incremental ingestion, large-scale transformations and optimized analytics, I would ingest the data into Bronze Delta and process it through Silver and Gold.

10. Final interview revision – Quick reference
Question	Feature / Concept
Hide customer email	Column Masking
Restrict Chennai orders	Row Filter
Query MySQL without copying	Lakehouse Federation
Govern ADLS/S3 storage	External Location
Organize external database metadata	Foreign Catalog
Troubleshoot access	GRANT / SHOW GRANTS
Trace Gold KPI	Data Lineage
Secure PostgreSQL credentials	Secrets + Connection
Historical and analytical processing	Delta Lake + Medallion